# Leakage-aware entity sentiment analysis

This notebook is the visual companion to the tested `sentiment_analysis` package. It profiles the data, demonstrates why a row-random split is unsafe, verifies the grouped boundary, and presents the checked-in experiment. Training lives in the CLI so notebook execution order cannot silently change the methodology.

## Setup

From the repository root, install the project with `python -m pip install -e ".[notebook]"`. The path logic below works whether Jupyter starts in the repository root or in `notebooks/`.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from sentiment_analysis.data import (
    LABELS,
    load_dataset,
    split_diagnostics,
    stratified_group_holdout,
)

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
assert (ROOT / "pyproject.toml").is_file(), "Start Jupyter in the repository or notebooks directory"
sns.set_theme(style="whitegrid", context="notebook")

## Validated data profile

The loader assigns the four documented columns, rejects unknown labels, trims text, removes unusable rows and exact duplicate rows, and fingerprints the input. Similar augmented variants remain in the data and are handled by grouping—not silently discarded.

In [ ]:
training, profile = load_dataset(ROOT / "data/twitter_training.csv")
pd.Series(profile.to_dict(), name="value")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.countplot(
    data=training,
    x="sentiment",
    order=LABELS,
    hue="sentiment",
    palette="Set2",
    legend=False,
    ax=axes[0],
)
axes[0].set(title="Usable records by sentiment", xlabel="", ylabel="Rows")
top_entities = training["entity"].value_counts().head(10).sort_values()
top_entities.plot.barh(ax=axes[1], color="#4C78A8")
axes[1].set(title="Ten most frequent entities", xlabel="Rows", ylabel="")
fig.tight_layout()

## Why the supplied validation file is not an independent holdout

The dataset is built around augmented families. The supplied validation records reuse training IDs, and many reuse exact training text. A row-level split therefore overstates generalization.

In [ ]:
validation, _ = load_dataset(ROOT / "data/twitter_validation.csv")
pd.Series(
    {
        "validation_rows": len(validation),
        "shared_tweet_ids": len(set(training["tweet_id"]) & set(validation["tweet_id"])),
        "shared_exact_texts": len(set(training["text"]) & set(validation["text"])),
    },
    name="overlap",
)

## Audited evaluation boundary

The package builds connected components over shared tweet IDs **and** exact text hashes. Each component stays entirely within one fold. This is stricter than grouping by ID alone because exact text can occur under different IDs.

In [ ]:
development, holdout = stratified_group_holdout(training, n_splits=5, random_state=42)
pd.Series(split_diagnostics(development, holdout), name="value")

## Canonical benchmark

`reports/benchmark.json` contains the five-fold grouped evaluation used throughout the project. Hyperparameters are selected inside each outer training partition, and every outer test fold remains isolated until evaluation.

In [ ]:
evidence = json.loads((ROOT / "reports/benchmark.json").read_text(encoding="utf-8"))
selected = evidence["candidates"][evidence["selection"]["model_kind"]]
pd.Series(
    {
        "accuracy_mean": selected["summary"]["accuracy"]["mean"],
        "accuracy_sd": selected["summary"]["accuracy"]["standard_deviation"],
        "macro_f1_mean": selected["summary"]["macro_f1"]["mean"],
        "macro_f1_sd": selected["summary"]["macro_f1"]["standard_deviation"],
    },
    name="score",
)

In [ ]:
labels = selected["folds"][0]["label_order"]
confusion = sum(
    (
        pd.DataFrame(fold["confusion_matrix"], index=labels, columns=labels)
        for fold in selected["folds"]
    ),
    start=pd.DataFrame(0, index=labels, columns=labels),
)
per_class_f1 = pd.Series(
    {
        label: sum(fold["classification_report"][label]["f1-score"] for fold in selected["folds"])
        / len(selected["folds"])
        for label in labels
    }
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(confusion, annot=True, fmt="d", cmap="Blues", ax=axes[0])
axes[0].set(title="Pooled outer-fold confusion matrix", xlabel="Predicted", ylabel="Actual")
per_class_f1.sort_values().plot.barh(ax=axes[1], color="#F58518", xlim=(0, 1))
axes[1].set(title="Mean per-class outer-fold F1", xlabel="F1", ylabel="")
fig.tight_layout()

## Interpretation

The grouped outer folds answer the credible question: how well does the model generalize to unseen duplicate families? Negative sentiment is strongest; `Irrelevant` remains the clearest improvement opportunity.

The final system preserves this boundary for classical model selection and transformer checkpoint selection. The README, benchmark reports, data card, and model card contain the reviewed results and deployment trade-offs.